# Perioperative timing and post-discharge SSI

This notebook regenerates the four timing comparisons in Section 3.1, the first four rows of Table 2, and Figures 2 and 5. Use the eight-week feature export and run this notebook before the final-model notebook so it can pair its reduced-model scores with the full-model scores.

Run through the predictor inventory cell to get the names needed for the clinical baseline and the temperature/WBC review. Model fitting starts in the next section.

In [ ]:
import json
import sys
from contextlib import redirect_stdout
from importlib.metadata import version
from io import StringIO
from pathlib import Path

import pandas as pd

N_JOBS = 2  # Parallel workers per random forest.

sys.path.insert(0, str(Path('../..').resolve()))
from ssi.feature_filter import feature_filter
from ssi.preprocess import preprocess

In [ ]:
df = pd.read_parquet(
    '../ssi_all_features.parquet',
    columns=['birthid', 'feature_name', 'feature_value'],
)
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''

with redirect_stdout(StringIO()):
    df_pivoted_preprocessed = preprocess(df_pivoted, defer_imputation=True)

excluded = ['birthid', 'mother_patid', 'target_variable']
X = df_pivoted_preprocessed.drop(columns=excluded)
y = df_pivoted_preprocessed['target_variable']
mother_patid = df_pivoted_preprocessed['mother_patid']

In [ ]:
# 1. All predictors immediately after preprocessing,
#    before the additional feature filtering.
preprocessed_feature_names = X.columns.tolist()

# 2. Predictors after the predefined SSI feature filtering.
with redirect_stdout(StringIO()):
    filtered_X = feature_filter(X.copy(), skip_medadmin_ingredient=True)

feature_names = filtered_X.columns.tolist()

# Validate both lists.
for label, names in {
    'preprocessed': preprocessed_feature_names,
    'available': feature_names,
}.items():
    if not names or len(names) != len(set(names)):
        raise ValueError(f'Expected a nonempty list of unique {label} predictors.')

# Save both lists.
inventory_dir = Path('results/feature_inventory')
inventory_dir.mkdir(parents=True, exist_ok=True)

preprocessed_path = (
    inventory_dir / 'ssi_preprocessed_feature_columns.txt'
)
available_path = (
    inventory_dir / 'ssi_available_feature_columns.txt'
)

preprocessed_path.write_text('\n'.join(preprocessed_feature_names) + '\n', encoding='utf-8')

available_path.write_text('\n'.join(feature_names) + '\n', encoding='utf-8')

print(f'Preprocessed predictors: {len(preprocessed_feature_names)}')
print(f'Filtered predictors: {len(feature_names)}')
print(f'Saved: {preprocessed_path}')
print(f'Saved: {available_path}')

## Four timing comparisons

The baseline uses medication exposure indicators. The other models add OR features, earliest medication administration times, or both. Each comparison uses the same mother-grouped splits and fits imputation and tuning on its training data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from src.clinical_imputer import ClinicalImputer
from src.model_training import nested_cv_evaluation_feature_selection

BASELINE = 'pcornet_has_med'
OR_TIMING = f'{BASELINE} + or_timings'
MED_TIMING = f'{BASELINE} + med_earliest_minute'
BOTH_TIMINGS = f'{MED_TIMING} + or_timings'

or_features = [name for name in feature_names if name.startswith('or_timings')]
ehr_features = [name for name in feature_names if not name.startswith('or_timings')]
baseline_features = [
    name for name in ehr_features
    if 'medadmin_ingredient' not in name or 'has_medadmin_ingredient' in name
]
med_timing_features = [
    name for name in ehr_features
    if 'medadmin_ingredient' not in name
    or 'has_medadmin_ingredient' in name
    or name.endswith('earliest_minute')
]
feature_sets = {
    BASELINE: baseline_features,
    OR_TIMING: baseline_features + or_features,
    MED_TIMING: med_timing_features,
    BOTH_TIMINGS: med_timing_features + or_features,
}

param_grid = {'clf__min_samples_split': [20, 50, 70, 100, 200, 500]}
cv_settings = dict(groups=mother_patid, outer_splits=5, inner_splits=5, random_state=range(10))

def rf_pipeline(columns):
    return Pipeline([
        ('clinical', ClinicalImputer(columns=columns)),
        ('clf', RandomForestClassifier(
            n_estimators=200, random_state=42, class_weight='balanced', n_jobs=N_JOBS,
        )),
    ])

In [ ]:
comparison_metrics = []
for name, columns in feature_sets.items():
    print(f'{name}: {len(columns)} predictors', flush=True)
    metrics = nested_cv_evaluation_feature_selection(
        X, y, rf_pipeline(columns), param_grid, **cv_settings,
    )
    comparison_metrics.append(metrics.assign(test=name))

all_metrics = pd.concat(comparison_metrics, ignore_index=True)

## Performance outputs and Figure 2

The performance summary compares the revised means with the rounded values from the submission. Paired differences use the same repeat and outer fold. Folds reuse mothers across repeats, so the outputs report descriptive differences and Figure 2 shows mean AUC.

In [ ]:
results_dir = Path('results/timing_revision')
results_dir.mkdir(parents=True, exist_ok=True)
all_metrics.to_csv(results_dir / 'outer_fold_metrics.csv', index=False)

run_config = {
    'rows': len(X),
    'ssi_events': int(y.sum()),
    'raw_engineered_predictors': sum(col not in excluded for col in df_pivoted.columns),
    'available_predictors': len(feature_names),
    'full_model_predictors': len(feature_sets[BOTH_TIMINGS]),
    'full_feature_names': feature_sets[BOTH_TIMINGS],
    'repeat_seeds': list(cv_settings['random_state']),
    'outer_splits': cv_settings['outer_splits'],
    'inner_splits': cv_settings['inner_splits'],
    'rf_grid': param_grid,
    'versions': {p: version(p) for p in ('numpy', 'pandas', 'scikit-learn')},
}
(results_dir / 'run_config.json').write_text(json.dumps(run_config, indent=2) + '\n')

submitted_auc = pd.Series({
    BASELINE: 0.677,
    OR_TIMING: 0.692,
    MED_TIMING: 0.693,
    BOTH_TIMINGS: 0.701,
}, name='submitted_roc_auc')
summary = all_metrics.groupby('test')[['roc_auc', 'ap', 'sens_80']].mean()
summary = summary.loc[submitted_auc.index].join(submitted_auc)
summary['roc_auc_change_from_submission'] = summary['roc_auc'] - summary['submitted_roc_auc']
summary.to_csv(results_dir / 'performance_summary.csv')
summary

In [ ]:
matched = all_metrics.pivot(index=['repeat', 'outer_fold'], columns='test', values='roc_auc')
if matched.isna().any().any():
    raise ValueError('Missing matched outer-fold ROC-AUC results.')
paired_deltas = matched[[OR_TIMING, MED_TIMING, BOTH_TIMINGS]].subtract(matched[BASELINE], axis=0)
paired_deltas['medication_timing_minus_or_timing'] = matched[MED_TIMING] - matched[OR_TIMING]
paired_deltas.to_csv(results_dir / 'paired_roc_auc_differences.csv')
paired_deltas.agg(['mean', 'std'])

In [ ]:
from matplotlib import pyplot as plt

labels = {
    BASELINE: 'Baseline EHR features',
    OR_TIMING: 'Baseline EHR +\nintraoperative timing features',
    MED_TIMING: 'Baseline EHR +\nmedication administration timing features',
    BOTH_TIMINGS: 'All features',
}
auc = summary.loc[list(labels), 'roc_auc']
fig, ax = plt.subplots()
ax.barh(list(labels.values()), auc)
ax.invert_yaxis()
ax.set_xlabel('ROC-AUC')
ax.set_xlim(min(0.60, auc.min() - 0.01), max(0.72, auc.max() + 0.01))
fig.savefig(results_dir / 'figure_2.png', bbox_inches='tight', dpi=200)
plt.show()

## Anesthesia and bupivacaine timing

Figure 5 and its correlations use encounters with both times recorded. Both axes measure minutes before delivery.

In [ ]:
anesthesia_col = 'or_timings_anes_start_datetime'
med_prefix = '1815_UNKNOWN_before_delivery'
bupivacaine_col = f'{med_prefix}_medadmin_ingredient_earliest_minute'
bupivacaine_exposure = f'{med_prefix}_has_medadmin_ingredient'

observed = pd.DataFrame({
    'anesthesia': -X[anesthesia_col].where(X[f'{anesthesia_col}_isna'].eq(0)),
    'bupivacaine': X[bupivacaine_col].where(X[bupivacaine_exposure].eq(1)),
}).dropna()
timing_summary = pd.Series({
    'observed_pairs': len(observed),
    'pearson_r': observed['anesthesia'].corr(observed['bupivacaine']),
    'spearman_rho': observed['anesthesia'].corr(observed['bupivacaine'], method='spearman'),
}, name='value')
timing_summary.to_csv(results_dir / 'figure_5_timing_summary.csv')

fig, ax = plt.subplots()
ax.scatter(observed['anesthesia'], observed['bupivacaine'])
ax.set(
    xlim=(-100, 3000), ylim=(-100, 3000),
    xlabel='Anesthesia start time (minutes before delivery)',
    ylabel='Earliest bupivacaine administration (minutes before delivery)',
)
fig.savefig(results_dir / 'figure_5.png', bbox_inches='tight', dpi=200)
plt.show()
timing_summary